In [ ]:
import pandas as pd
import numpy as np
import os
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.metrics import classification_report
# Suppress TensorFlow logging warnings for a cleaner output
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

## 1. Data Loading and Preparation

In [4]:
# Read the CSV files
train_df = pd.read_csv('sent_train.csv')
valid_df = pd.read_csv('sent_valid.csv')

# Drop any rows with missing values to prevent training errors
train_df.dropna(inplace=True)
valid_df.dropna(inplace=True)

# Separate the input features (text) and the target labels
X_train = train_df['text'].astype(str)
y_train = train_df['label'].values
X_valid = valid_df['text'].astype(str)
y_valid = valid_df['label'].values

## 2. Text Preprocessing

In [6]:
# Set up hyperparameters for tokenization and padding
vocab_size = 10000  # Only consider the top 10,000 most frequent words
max_length = 50     # Limit the length of each tweet to 50 words
trunc_type = 'post' # Truncate sequences longer than 50 words at the end
padding_type = 'post' # Pad sequences shorter than 50 words at the end
oov_tok = "<OOV>"   # Out-of-vocabulary token for words not in the index

# Initialize the Tokenizer and fit it strictly on the training data
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

# Convert the text into sequences of integers
train_sequences = tokenizer.texts_to_sequences(X_train)
# Pad the sequences so they are all the same length for the neural network
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

# Apply the same transformations to the validation set
valid_sequences = tokenizer.texts_to_sequences(X_valid)
valid_padded = pad_sequences(valid_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

## 3. Defining the LSTM Architecture

In [7]:
embedding_dim = 64 # Dimensions of the word embeddings

model = Sequential([
    # Embedding layer to learn dense vector representations for words
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    
    # First LSTM layer returning sequences to feed into the next LSTM layer
    LSTM(64, return_sequences=True),
    # Dropout layer to prevent overfitting by randomly disabling 50% of neurons
    Dropout(0.5),
    
    # Second LSTM layer summarizing the sequence
    LSTM(32),
    
    # Fully connected Dense layer for intermediate processing
    Dense(32, activation='relu'),
    Dropout(0.5), # Additional dropout for regularization
    
    # Output layer with 3 units and softmax activation for probabilities
    Dense(3, activation='softmax') 
])

# Compile the model using Sparse Categorical Crossentropy since labels are integers (0, 1, 2)
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

c:\Users\ybual\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


## 4. Model Training

In [9]:
num_epochs = 10
# Train the model, storing the training history, and validating simultaneously
history = model.fit(
    train_padded, y_train, 
    epochs=num_epochs, 
    validation_data=(valid_padded, y_valid), 
    verbose=1
)

Epoch 1/10
299/299 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.6474 - loss: 0.8881 - val_accuracy: 0.6558 - val_loss: 0.8744
Epoch 2/10
299/299 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.6474 - loss: 0.8885 - val_accuracy: 0.6558 - val_loss: 0.8755
Epoch 3/10
299/299 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.6474 - loss: 0.8896 - val_accuracy: 0.6558 - val_loss: 0.8756
Epoch 4/10
299/299 ━━━━━━━━━━━━━━━━━━━━ 14s 47ms/step - accuracy: 0.6474 - loss: 0.8886 - val_accuracy: 0.6558 - val_loss: 0.8749
Epoch 5/10
299/299 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.6474 - loss: 0.8883 - val_accuracy: 0.6558 - val_loss: 0.8797
Epoch 6/10
299/299 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.6474 - loss: 0.8909 - val_accuracy: 0.6558 - val_loss: 0.8784
Epoch 7/10
299/299 ━━━━━━━━━━━━━━━━━━━━ 12s 39ms/step - accuracy: 0.6474 - loss: 0.8898 - val_accuracy: 0.6558 - val_loss: 0.8783
Epoch 8/10
299/299 ━━━━━━━━━━━━━━━━━━━━ 13s 44ms/step - accuracy: 0.6474 - loss: 0.8914 - va

## 5. Model Evaluation and Validation

In [10]:
# Get the overall validation loss and accuracy
loss, accuracy = model.evaluate(valid_padded, y_valid, verbose=0)
print(f"Overall Validation Accuracy: {accuracy:.4f}\n")

# Get model predictions to generate a detailed classification report
y_pred_probs = model.predict(valid_padded, verbose=0)
# Convert probabilities to class labels using argmax
y_pred = np.argmax(y_pred_probs, axis=1)

# Map labels to their sentiment names for a readable report
target_names = ['Bearish (0)', 'Bullish (1)', 'Neutral (2)']
print("Detailed Classification Report:")
print(classification_report(y_valid, y_pred, target_names=target_names))

Overall Validation Accuracy: 0.6558

Detailed Classification Report:
              precision    recall  f1-score   support

 Bearish (0)       0.00      0.00      0.00       347
 Bullish (1)       0.00      0.00      0.00       475
 Neutral (2)       0.66      1.00      0.79      1566

    accuracy                           0.66      2388
   macro avg       0.22      0.33      0.26      2388
weighted avg       0.43      0.66      0.52      2388



c:\Users\ybual\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ybual\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ybual\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave